# PhishGuard Anomaly Detection - LANL Dataset Training

This notebook trains a PyTorch LSTM Autoencoder on the official Los Alamos National Laboratory (LANL) Cybersecurity dataset.

**Hardware Check:**
Ensure you are using a T4 GPU runtime: `Runtime` -> `Change runtime type` -> Hardware accelerator: `T4 GPU`.

In [ ]:
!pip install onnx onnxscript pandas
!nvidia-smi

## 1. Import Dependencies

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 2. Load LANL Dataset

**IMPORTANT:** You must download `auth.txt.gz` from the [LANL Cybersecurity portal](https://csr.lanl.gov/data/cyber1/). 
Upload it to Google Colab using the file explorer on the left, or mount your Google Drive.

In [ ]:
import os
if not os.path.exists('auth.txt.gz'):
    print("\u26A0\uFE0F WARNING: auth.txt.gz not found!")
    print("Please download the dataset from https://csr.lanl.gov/data/cyber1/")
    print("Upload it here, then rerun this cell.")
else:
    print("\u2705 Found auth.txt.gz! Reading data (this may take a minute)...")
    
    # LANL auth.txt columns
    cols = ['time', 'source_user', 'dest_user', 'source_computer', 'dest_computer', 
            'auth_type', 'logon_type', 'auth_orientation', 'success']
    
    # We only take the first 500,000 lines to fit easily in Colab RAM
    df = pd.read_csv('auth.txt.gz', names=cols, usecols=['time', 'source_computer', 'success'], nrows=500000)
    
    # Keep only successful normal logins to train the Autoencoder baseline
    df = df[df['success'] == 'Success'].copy()
    print(f"Loaded {len(df)} normal login events.")

## 3. Map LANL Data to PhishGuard's 8 Dimensions
We extract time components directly from the LANL timeline. Since corporate LANL logs do not record exact geographical coordinates or typing speed, we synthesize the missing biometric/spatial dimensions to match the PhishGuard `LoginAutoencoder` input.

In [ ]:
if 'df' in locals():
    print("Processing feature vectors...")
    # 1. Hour of Day (LANL time is in seconds. 3600 sec = 1 hr)
    df['hour_of_day'] = (df['time'] // 3600) % 24
    
    # 2. Day of Week (LANL time: 86400 sec = 1 day)
    df['day_of_week'] = (df['time'] // 86400) % 7
    
    # 3. Failures in last hour (Simplified mapping for autoencoder normal baseline)
    df['failures_last_hour'] = 0.0 
    
    # 4. IP is new (Track rolling seen counts per computer)
    df['ip_is_new'] = (~df.duplicated(subset=['source_computer'])).astype(float)
    
    # 5. Device is new 
    df['device_is_new'] = df['ip_is_new'] * 0.5  # Approximate correlation
    
    # 6. Geo distance km (Synthesized logic for normal corporate behavior)
    df['geo_distance_km'] = np.random.uniform(0, 15, size=len(df))
    
    # 7. Time since last login hrs (Calculate difference for each computer)
    df = df.sort_values(by=['source_computer', 'time'])
    df['time_since_last_login_hrs'] = df.groupby('source_computer')['time'].diff().fillna(86400) / 3600
    # Clamp unrealistic large gaps
    df['time_since_last_login_hrs'] = df['time_since_last_login_hrs'].clip(upper=48.0)
    
    # 8. Typing speed ms (Synthesized biometric cadence for normal users)
    df['typing_speed_ms'] = np.random.normal(100, 20, size=len(df)).clip(min=30)
    
    # Create final numpy array
    features = ['hour_of_day', 'day_of_week', 'failures_last_hour', 'ip_is_new', 
                'device_is_new', 'geo_distance_km', 'time_since_last_login_hrs', 'typing_speed_ms']
    
    X_train = df[features].values.astype(np.float32)
    print(f"X_train shape generated from LANL: {X_train.shape}")
else:
    print("\u26A0\uFE0F Load the dataset first!")

## 4. Normalization & Data Loading

In [ ]:
if 'X_train' in locals():
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-8
    X_norm = (X_train - mean) / std

    # Model expects sequence format: (batch, seq_len=1, features)
    X_tensor = torch.FloatTensor(X_norm).unsqueeze(1)

    batch_size = 512
    dataset = TensorDataset(X_tensor, X_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    print("DataLoader ready.")

## 5. Define PyTorch LSTM Autoencoder

In [ ]:
class LoginAutoencoder(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64, latent_dim=16, n_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.enc_fc = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, input_dim, n_layers, batch_first=True)

    def forward(self, x):
        enc_out, _ = self.encoder(x)
        latent = self.enc_fc(enc_out[:, -1, :])
        dec_input = self.dec_fc(latent).unsqueeze(1).repeat(1, x.size(1), 1)
        dec_out, _ = self.decoder(dec_input)
        return dec_out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = LoginAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 6. Training Loop

In [ ]:
if 'loader' in locals():
    epochs = 30
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch + 1}/{epochs}] Loss: {total_loss / len(loader):.6f}")
    print("Training complete! \u2705")

## 7. Export Artifacts (ONNX + NPZ)

In [ ]:
if 'X_train' in locals():
    model.eval()
    dummy_input = torch.randn(1, 1, 8).to(device)
    onnx_path = "login_autoencoder.onnx"

    torch.onnx.export(
        model, dummy_input, onnx_path,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
    )

    npz_path = "norm_params.npz"
    np.savez(npz_path, mean=mean, std=std)

    print(f"\u2728 Saved ONNX model to {onnx_path}")
    print(f"\u2728 Saved Normalization parameters to {npz_path}")

## 8. Download to Local Machine
Run this cell to force the browser to download the required files so you can copy them to `backend/ml/`

In [ ]:
try:
    from google.colab import files
    files.download('login_autoencoder.onnx')
    files.download('norm_params.npz')
except ImportError:
    print("Not running in Google Colab. Files are saved in the current directory.")
except Exception as e:
    print(f"Download skipped. Model is ready on Colab disk.")